# 01 - Data Validation & Cleaning

**Goal:** load the raw Kaggle Retail Store Inventory dataset, run data-quality
checks, and produce a cleaned dataset ready for EDA and feature engineering.

Dataset: `data/raw/retail_store_inventory.csv` (73,100 rows: 5 stores x 20 products x 731 days, Jan 2022 - Jan 2024).


In [13]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from src.data_processing import load_raw, validate_raw, clean

pd.set_option('display.max_columns', None)
raw = load_raw()
raw.shape

(73100, 15)

In [14]:
raw.head()

,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


In [15]:
raw.dtypes

Date                   object
Store ID               object
Product ID             object
Category               object
Region                 object
Inventory Level         int64
Units Sold              int64
Units Ordered           int64
Demand Forecast       float64
Price                 float64
Discount                int64
Weather Condition      object
Holiday/Promotion       int64
Competitor Pricing    float64
Seasonality            object
dtype: object

## Data quality checks

We check: nulls, duplicates, negative values in numeric columns, whether the
panel (store x product x date) is a complete rectangle (required for lag/rolling
features to be correct later), and whether units sold ever exceeds inventory.

In [16]:
report = validate_raw(raw)
for k, v in report.items():
    print(f"{k}: {v}")

n_rows: 73100
n_duplicates: 0
missing_by_col: {'Date': 0, 'Store ID': 0, 'Product ID': 0, 'Category': 0, 'Region': 0, 'Inventory Level': 0, 'Units Sold': 0, 'Units Ordered': 0, 'Demand Forecast': 0, 'Price': 0, 'Discount': 0, 'Weather Condition': 0, 'Holiday/Promotion': 0, 'Competitor Pricing': 0, 'Seasonality': 0}
negative_value_counts: {'Demand Forecast': 673}
panel_is_balanced: True
rows_per_combo: 731
sold_exceeds_inventory: 0


## Findings

- **No missing values, no duplicate rows.** Clean source data.
- **`Demand Forecast` has 673 negative values (~0.9%).** This is a vendor-supplied
  legacy forecast column, not our target. Negative forecasted demand is
  physically meaningless, which tells us this column is noisy and should NOT
  be used as a model feature (see reasoning in `src/data_processing.py`) -- we
  keep it only as a benchmark to compare our model against later.
- **Panel is a perfect rectangle**: every one of the 100 store-product
  combinations has exactly 731 rows (one per day, no gaps). This matters a lot
  -- lag/rolling features assume no missing days, and this dataset gives us that
  for free.
- **Units sold never exceeds inventory level** on the same row -- internally
  consistent, no impossible business values found.

## Cleaning decisions applied (see `src/data_processing.py::clean` for full reasoning)
1. Rename raw columns to snake_case.
2. Parse `Date`, sort by (store, product, date) -- required for correct lag/rolling features.
3. Drop exact duplicate rows (0 found, kept as a safeguard).
4. Do **not** clip or remove outliers in `units_sold` / `inventory_level` -- retail
   demand is naturally spiky (promotions, holidays); removing those points would
   strip out exactly the signal the model needs to learn.
5. Cast categorical columns to `category` dtype.


In [17]:
clean_df = clean(raw)
clean_df.head()

,date,store_id,product_id,category,region,inventory_level,units_sold,units_ordered,legacy_demand_forecast,price,discount_pct,weather,promotion,competitor_price,seasonality_label
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-02,S001,P0001,Groceries,West,116,81,104,92.94,27.95,10,Cloudy,0,30.89,Spring
2,2022-01-03,S001,P0001,Electronics,West,154,5,189,5.36,62.70,20,Rainy,0,58.22,Winter
3,2022-01-04,S001,P0001,Groceries,South,85,58,193,52.87,77.88,15,Cloudy,1,75.99,Winter
4,2022-01-05,S001,P0001,Groceries,South,238,147,37,150.27,28.46,20,Sunny,1,29.40,Winter


In [18]:
from src.utils import PROCESSED_DATA_PATH
clean_df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"Saved cleaned dataset: {clean_df.shape} -> {PROCESSED_DATA_PATH}")

Saved cleaned dataset: (73100, 15) -> c:\Users\Pdllab\Downloads\demand-forecasting-inventory-optimization (2)\demand-forecasting-inventory-optimization\data\processed\clean_sales.csv
